# V3 / 00 — UAV input inventory and spatial data audit

**Start here.** CPU-only, Google Colab + Drive. This notebook discovers the authoritative input files in `Musanga/Final/Data`, inventories native RGB orthomosaics, DSM, ROI and tree annotations, and verifies their spatial relationships. It does **not** rewrite, reproject, or copy input data.

**V3 data contract:** `Final/Data/Ortho_RGB/` holds six native RGB orthomosaics and the modelling ROI shapefile (or ROI may be in the `Final/Data/` root); `Final/Data/DSM/` currently contains only Baego DSM; `Final/Data/Annotations/` holds tree-annotation shapefiles. **No Luki**. MOS is **one site with three distinct mosaics** and **one shared crown annotation inventory**. Report these separately.

Outputs: small audit CSV and JSON files in `Final/Outputs/V3/00_input_audit/`. No training or probability inference in Notebook 00. Assumptions about labels (for example, `Class_ID==1`) are **inspected, not silently imposed**; modelling class mapping is a later V3 decision.

**New exhaustive labels:** `Annotations/Circles_Annotations.shp` identifies Musanga-positive polygon areas within `Annotations/Circles.shp` boundaries. Unannotated area *inside those circle ROIs* is confirmed non-Musanga; everything outside is unlabelled. This inventory is distinct from per-site, partially annotated tree crowns.


## 01 — Environment and Drive setup

Mount Google Drive, locate the authoritative `Final/Data` folders, and establish the audit output location. Original data are never modified.


In [ ]:
# V3 Notebook 00 / Block 01 — Mount Drive and locate the FINAL data directory
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib
from pathlib import Path
import json, re, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.validation import explain_validity
from IPython.display import display
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
MYDRIVE=Path("/content/drive/MyDrive")

# Set this manually only if your MyDrive location is not among these two:
DATA_ROOT_OVERRIDE=None
CANDIDATE_DATA_ROOTS=[
    MYDRIVE/"Colab Notebooks/Musanga/Final/Data",
    MYDRIVE/"Musanga/Final/Data",
]
DATA=Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else next(
    (p for p in CANDIDATE_DATA_ROOTS if p.is_dir()),None)
assert DATA is not None,(
    "Cannot find Final/Data. Set DATA_ROOT_OVERRIDE above to the actual "
    "Google Drive path. Tried: "+", ".join(map(str,CANDIDATE_DATA_ROOTS)))
RGB_DIR=DATA/"Ortho_RGB"
DSM_DIR=DATA/"DSM"
ANN_DIR=DATA/"Annotations"
for d in (RGB_DIR,DSM_DIR,ANN_DIR):
    assert d.is_dir(),f"Required folder missing: {d}"
OUT=DATA.parent/"Outputs"/"V3"/"00_input_audit"
OUT.mkdir(parents=True,exist_ok=True)
print("Authoritative data directory:",DATA)
print("Audit reports (only writes here):",OUT)
print("Input subfolders:")
for d in (RGB_DIR,DSM_DIR,ANN_DIR):
    print(" ",d.name,":",len(list(d.iterdir())),"entries")


## 02 — Native RGB orthomosaics and DSM inventory

Identify six orthomosaics across four site groups, including the three MOS mosaics, and inspect native resolution, CRS, band structure and DSM availability.


In [ ]:
# V3 Notebook 00 / Block 02 — Native image registry (no imagery is loaded)
def tifs(folder):
    return sorted([p for p in folder.iterdir() if p.is_file()
                   and p.suffix.lower() in (".tif",".tiff")],
                  key=lambda p:p.name.lower())
rgb_paths=tifs(RGB_DIR)
dsm_paths=tifs(DSM_DIR)
print("RGB mosaics found:",len(rgb_paths))
for p in rgb_paths:print("  ",p.name)
print("DSM files found:",len(dsm_paths))
for p in dsm_paths:print("  ",p.name)

# Matching uses filenames only to assign site; actual basenames are NOT assumed.
# No Luki in V3: fail if unexpected files are included.
def assign_site(name):
    x=name.lower()
    if "luki" in x:return "Luki"
    if "baeg" in x:return "Baego"
    if "yangambi" in x:return "Yangambi"
    if "yoko" in x:return "Yoko"
    if re.search(r"(^|[^a-z])mos([_\-.]|$)",x):return "MOS"
    return "UNRECOGNIZED"
def mos_group(name):
    m=re.search(r"(?<!\d)([135])[_-]([246])(?!\d)",name.lower())
    return f"MOS_{m.group(1)}_{m.group(2)}" if m else None

entries=[];raster_bounds={}
for kind,paths in (("RGB",rgb_paths),("DSM",dsm_paths)):
    for p in paths:
        site=assign_site(p.stem)
        try:
            with rasterio.open(p) as src:
                r=dict(role=kind,site=site,mos_group=mos_group(p.stem) if site=="MOS" else "",
                       filename=p.name,path=str(p),size_GB=round(p.stat().st_size/1e9,3),
                       width=src.width,height=src.height,bands=src.count,
                       dtype_first3="/".join(src.dtypes[:3]),
                       crs=str(src.crs),pixel_x=abs(src.res[0]),pixel_y=abs(src.res[1]),
                       pixel_area_m2=abs(src.transform.a*src.transform.e-src.transform.b*src.transform.d),
                       nodata=str(src.nodata),
                       bounds=str(tuple(round(v,2) for v in src.bounds)),
                       colorinterp="/".join(str(v) for v in src.colorinterp[:3]),
                       error="")
                raster_bounds[p.name]=(src.crs,box(*src.bounds))
                if not src.crs: r["error"]="Missing CRS"
                elif kind=="RGB" and src.count<3:r["error"]="RGB requires at least 3 bands"
                elif kind=="RGB" and src.count>3:
                    print("NOTE:",p.name,"has",src.count,"bands; V3 RGB uses first 3 only")
        except Exception as exc:
            r=dict(role=kind,site=site,filename=p.name,path=str(p),
                   error=f"{type(exc).__name__}: {exc}")
        entries.append(r)
rasters=pd.DataFrame(entries)
display(rasters)
rgb=rasters[rasters.role=="RGB"]
print("RGB mosaic counts per site:",rgb.site.value_counts().to_dict())
assert len(rgb_paths)==6, f"Expected 6 RGB orthomosaics; found {len(rgb_paths)}"
assert rgb.site.value_counts().to_dict()=={"MOS":3,"Baego":1,"Yangambi":1,"Yoko":1},(
    "Unexpected site distribution or naming; inspect the table and update assign_site.")
assert not rasters.error.fillna("").astype(str).str.len().gt(0).any(), "Raster metadata errors in table"
if len(dsm_paths)!=1 or assign_site(dsm_paths[0].stem)!="Baego":
    print("NOTE: DSM availability differs from expected single Baego DSM.")
rasters.to_csv(OUT/"00_raster_inventory.csv",index=False)


## 03 — Crown annotations and general modelling ROI

Inventory the partially annotated tree-crown files, check label attributes and geometry integrity, and locate **`ROI.shp`** (the general modelling area, not the exhaustive-circle boundary file).


In [ ]:
# V3 Notebook 00 / Block 03 — Annotation registry, label QA and ROI discovery
# Annotation filenames may vary (plural Tree_Annotations or singular Tree_Annotation).
vector_exts={".shp",".gpkg",".geojson"}
ann_files=sorted([p for p in ANN_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in vector_exts],key=lambda p:p.name.lower())
print("Annotation vector files:",len(ann_files))
for p in ann_files:print(" ",p.relative_to(ANN_DIR))

# Prefer ROI.shp within Ortho_RGB or directly under Final/Data. Search only under Final/Data.
roi_candidates=sorted([p for p in DATA.rglob("*") if p.is_file()
                       and p.suffix.lower() in vector_exts
                       and (p.stem.lower()=="roi" or p.stem.lower().startswith("roi_"))],
                      key=lambda p:(len(p.parts),str(p)))
print("ROI candidates:",[str(p.relative_to(DATA)) for p in roi_candidates])
assert len(roi_candidates)==1, (
    "Exactly one ROI vector is expected. Set an explicit ROI path if multiple candidates appear.")
ROI_FILE=roi_candidates[0]
def inspect_vector(path,role):
    records=[]
    try:
        layers=[None]
        if path.suffix.lower()==".gpkg":
            import fiona
            layers=fiona.listlayers(path)
        for layer in layers:
            g=gpd.read_file(path,layer=layer) if layer is not None else gpd.read_file(path)
            bad=int((~g.geometry.is_valid).sum())
            missing=int(g.geometry.isna().sum())
            record=dict(role=role,site=assign_site(path.stem),
                 file=path.name,path=str(path),layer=layer or "",
                 n_features=len(g),crs=str(g.crs),geom_types=";".join(sorted(map(str,g.geom_type.unique()))),
                 invalid_geometries=bad,null_geometries=missing,
                 fields=";".join(map(str,g.columns)),
                 label_field="Class_ID" if "Class_ID" in g.columns else
                    ("Espece" if "Espece" in g.columns else
                     ("Nom" if "Nom" in g.columns else "")),
                 error="")
            records.append(record)
    except Exception as exc:
        records.append(dict(role=role,site=assign_site(path.stem),file=path.name,
                            path=str(path),error=f"{type(exc).__name__}: {exc}"))
    return records

ann_records=[]
for f in ann_files:ann_records+=inspect_vector(f,"tree_annotations")
roi_records=inspect_vector(ROI_FILE,"ROI")
ann_table=pd.DataFrame(ann_records)
display(ann_table)
display(pd.DataFrame(roi_records))
assert not any(r["error"] for r in ann_records+roi_records), "Vector read failures: inspect table"
ann_table.to_csv(OUT/"00_annotation_inventory.csv",index=False)
pd.DataFrame(roi_records).to_csv(OUT/"00_roi_inventory.csv",index=False)

# Class frequencies are descriptive; do not assume all datasets use identical coding.
freq=[]
for f in ann_files:
    g=gpd.read_file(f)
    field=next((k for k in ("Class_ID","Espece","Nom") if k in g.columns),None)
    if field is not None:
        for value,count in g[field].fillna("<MISSING>").astype(str).value_counts().items():
            freq.append(dict(file=f.name,site=assign_site(f.stem),
                             label_field=field,value=value,count=int(count)))
if freq:
    display(pd.DataFrame(freq))
    pd.DataFrame(freq).to_csv(OUT/"00_annotation_label_counts.csv",index=False)

# Check mandatory shapefile sidecars; absence of .prj means geographic alignment is unknown.
missing_sidecars=[]
for f in [ROI_FILE]+[x for x in ann_files if x.suffix.lower()==".shp"]:
    for suffix in (".shx",".dbf",".prj"):
        if not f.with_suffix(suffix).is_file():
            missing_sidecars.append(str(f.with_suffix(suffix)))
print("Missing shapefile sidecars:",missing_sidecars or "none")
assert not missing_sidecars,"Fix incomplete shapefile sets before modelling."


## 04 — Spatial consistency and shared MOS annotations

Check alignment between each orthomosaic, its per-site crown annotations and the general ROI. The three MOS mosaics share one annotation dataset.


In [ ]:
# V3 Notebook 00 / Block 04 — Associate MOS mosaics with ONE shared annotation file
# Compare vector and raster footprint intersections in the native raster CRS.
from shapely.ops import unary_union
ann_site={}
for site in ("Baego","Yangambi","Yoko","MOS"):
    matches=[p for p in ann_files if assign_site(p.stem)==site
             and "tree" in p.stem.lower()]
    print(site,": annotation matches:",[p.name for p in matches])
    assert len(matches)==1,(
        f"Expected one tree-annotation inventory for {site}; found {len(matches)}. "
        "Inspect names or define an explicit mapping.")
    ann_site[site]=matches[0]
roi_g=gpd.read_file(ROI_FILE)
assert roi_g.crs is not None,"ROI has no CRS"
roi_valid=roi_g.geometry.dropna()
assert len(roi_valid) and roi_valid.is_valid.all(),"ROI has missing/invalid geometry"
roi_geom=unary_union(roi_valid.to_list())
print("ROI:",ROI_FILE.name,"features",len(roi_g),"CRS",roi_g.crs)

spatial=[]
for p in rgb_paths:
    site=assign_site(p.stem)
    with rasterio.open(p) as src:
        footprint=box(*src.bounds)
        annotations=gpd.read_file(ann_site[site])
        assert annotations.crs is not None,f"Missing annotation CRS: {ann_site[site].name}"
        ann_aligned=annotations.to_crs(src.crs)
        # Only bounds-test per feature, not an expensive polygon union.
        intersects=ann_aligned.geometry.intersects(footprint)
        roi_aligned=roi_g.to_crs(src.crs)
        roi_intersect=unary_union(roi_aligned.geometry.to_list()).intersection(footprint).area
        footprint_area=footprint.area
        spatial.append(dict(site=site,mosaic=p.name,
            annotation_file=ann_site[site].name,
            annotation_total=len(annotations),
            annotation_intersecting_mosaic=int(intersects.sum()),
            annotation_outside_mosaic=int((~intersects).sum()),
            roi_mosaic_intersection_m2=roi_intersect,
            roi_fraction_of_mosaic_bbox=roi_intersect/max(footprint_area,1),
            imagery_crs=str(src.crs),annotations_crs=str(annotations.crs),
            roi_crs=str(roi_g.crs)))
relations=pd.DataFrame(spatial)
display(relations)
relations.to_csv(OUT/"00_mosaic_annotation_roi_relationships.csv",index=False)
print("MOS shared annotation file:",ann_site["MOS"].name)
print("MOS annotation records are NOT duplicated into three source files.")
if (relations.annotation_intersecting_mosaic==0).any():
    warnings.warn("One or more mosaic/annotation pairs have no spatial intersection. "
                  "Check naming, coordinate systems and annotation coverage.")
if (relations.roi_mosaic_intersection_m2==0).any():
    warnings.warn("One or more mosaics have no ROI overlap; inspect ROI CRS and location.")

# Lightweight overview figure from geometry, no large UAV pixel reads.
fig,ax=plt.subplots(figsize=(9,8))
for _,r in relations.iterrows():
    crs,geom=raster_bounds[r.mosaic]
    display_geom=gpd.GeoSeries([geom],crs=crs).to_crs(roi_g.crs).iloc[0]
    x,y=display_geom.exterior.xy
    ax.plot(x,y,label=r.site+(" "+str(mos_group(r.mosaic)) if r.site=="MOS" else ""))
gpd.GeoSeries([roi_geom],crs=roi_g.crs).boundary.plot(ax=ax,color="black",linewidth=2)
ax.set_title("RGB mosaic bounding boxes and model ROI (same CRS)")
ax.set_aspect("equal");ax.legend(fontsize=8)
fig.savefig(OUT/"00_mosaic_roi_overview.png",dpi=150,bbox_inches="tight")
plt.show()


## 05 — Exhaustive Musanga circles

Audit **`Circles_Annotations.shp`** (Musanga-positive polygons) and **`Circles.shp`** (exhaustively labelled circle boundaries). Inside circles but outside the Musanga polygons is confirmed non-Musanga; outside circles is unlabelled. Check geometry, areas, coverage and which mosaics contain the circles.


In [ ]:
# V3 Notebook 00 / Block 05 — Exhaustive Musanga circle annotations (separate supervision type)
# Confirmed names: Circles_Annotations.shp = positive Musanga polygons;
# Circles.shp = boundaries with exhaustive annotation (distinct from general ROI.shp).
from shapely.ops import unary_union
from shapely.geometry import Polygon,MultiPolygon
from shapely import make_valid

CIRCLE_LABELS=ANN_DIR/"Circles_Annotations.shp"
CIRCLE_ROIS=ANN_DIR/"Circles.shp"
print("Circle-related shapefiles available in Annotations:")
for candidate in sorted(ANN_DIR.rglob("*.shp")):
    if "circle" in candidate.name.lower():
        print(" ",candidate.relative_to(ANN_DIR))
print("Exhaustive annotation paths:")
for role,p in [("Musanga crowns",CIRCLE_LABELS),("complete-annotation boundaries",CIRCLE_ROIS)]:
    print(" ",role,":",p,"exists:",p.is_file())
    assert p.is_file(),f"Missing required exhaustive annotation input: {p}"
    for ext in (".shx",".dbf",".prj"):
        assert p.with_suffix(ext).is_file(),f"Incomplete shapefile: {p.with_suffix(ext)}"

def polygon_parts00(g):
    if isinstance(g,Polygon):return [g]
    if isinstance(g,MultiPolygon):return list(g.geoms)
    if hasattr(g,"geoms"):
        return [p for q in g.geoms for p in polygon_parts00(q)]
    return []

def inspect_and_repair00(path):
    raw=gpd.read_file(path)
    assert raw.crs is not None,f"{path.name} has no CRS"
    assert len(raw)>0 and raw.geometry.notna().all()
    bad=~raw.geometry.is_valid
    repaired=[]
    changes=[]
    for i,g in enumerate(raw.geometry):
        assert not g.is_empty,f"{path.name} empty geometry at row {i}"
        if not g.is_valid:
            parts=polygon_parts00(make_valid(g))
            assert parts,f"{path.name}: row {i} cannot be made into polygon geometry"
            fixed=unary_union(parts)
            change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
            changes.append((i,change))
            assert change<=.01,(f"{path.name} row {i}: repair changes area by {change:.1%}, "
                                 "please inspect the source")
            g=fixed
        repaired.append(g)
    out=raw.copy()
    out[out.geometry.name]=repaired
    assert out.geometry.is_valid.all()
    assert out.geom_type.isin(["Polygon","MultiPolygon"]).all(),(
        f"{path.name}: expected polygon geometries")
    print(f"{path.name}: {len(raw)} features, {int(bad.sum())} invalid "
          f"(in-memory repairs only), CRS={raw.crs}")
    if changes:
        print(" Largest relative area changes:",sorted(changes,key=lambda x:-x[1])[:6])
    return out

exhaustive_rois00=inspect_and_repair00(CIRCLE_ROIS)
exhaustive_crowns00=inspect_and_repair00(CIRCLE_LABELS)
# Reproject vectors ONLY in memory to the boundary CRS; preserve source files.
crowns_aligned00=exhaustive_crowns00.to_crs(exhaustive_rois00.crs)
circle_union00=unary_union(exhaustive_rois00.geometry.to_list())
crown_union00=unary_union(crowns_aligned00.geometry.to_list())
contained00=crown_union00.intersection(circle_union00)
assert contained00.area>0,"Exhaustive Musanga polygons do not intersect Circles_ROI boundaries"
coverage00=contained00.area/crown_union00.area
print(f"Annotated Musanga area inside exhaustive ROI union: {coverage00:.2%}")
print("Musanga polygon area outside circles (ignored):",
      round(crown_union00.area-contained00.area,2),"CRS area units squared")
if coverage00<.85:
    warnings.warn("Less than 85% of annotated Musanga polygon area falls inside circles; "
                  "verify these are matching ROI and crown files.")

circle_rows00=[]
for i,geom in enumerate(exhaustive_rois00.geometry):
    pos=contained00.intersection(geom)
    circle_rows00.append(dict(circle_row=i,
        circle_area_units2=float(geom.area),
        musanga_area_units2=float(pos.area),
        fraction_musanga=float(pos.area/geom.area) if geom.area else np.nan,
        n_crowns_intersecting=int(crowns_aligned00.geometry.intersects(geom).sum())))
circle_summary00=pd.DataFrame(circle_rows00)
display(circle_summary00)
circle_summary00.to_csv(OUT/"00_exhaustive_circle_area_audit.csv",index=False)

# Determine which mosaic(s) overlap each exhaustive circle; no assumption
# that this dataset is Yangambi-only or that all mosaic CRS are identical.
relations00=[]
for circle_idx,g in enumerate(exhaustive_rois00.geometry):
    for image_path in rgb_paths:
        with rasterio.open(image_path) as src:
            geom_reprojected=gpd.GeoSeries([g],crs=exhaustive_rois00.crs).to_crs(src.crs).iloc[0]
            footprint=box(*src.bounds)
            intersection=float(geom_reprojected.intersection(footprint).area)
            if intersection>0:
                relations00.append(dict(circle_row=circle_idx,
                    mosaic=image_path.name,site=assign_site(image_path.stem),
                    intersecting_area_mosaic_crs_units2=intersection,
                    circle_fraction_in_mosaic_bbox=intersection/max(geom_reprojected.area,1e-12)))
relations_df00=pd.DataFrame(relations00)
print("Exhaustive circle ↔ RGB mosaic overlaps:")
display(relations_df00)
relations_df00.to_csv(OUT/"00_exhaustive_circle_mosaic_links.csv",index=False)
if len(relations00)==0:
    warnings.warn("No exhaustive circles overlap any RGB mosaic; investigate CRS and georeferencing.")

print("Supervision contract: Musanga pixels are positive INSIDE the circles;")
print("unannotated pixels are confirmed non-Musanga ONLY INSIDE the circles.")
print("Outside circles: UNLABELLED (not negative).")


## 06 — Save the V3 data contract

Save reproducible source paths, spatial QA tables and the exhaustive-annotation supervision rules for the next V3 notebooks.


In [ ]:
# V3 Notebook 00 / Block 06 — Persistent input manifest and next-step checks
manifest=dict(schema_version="v3_input_audit_2",
    data_root=str(DATA),output_root=str(OUT),
    site_groups=["Baego","MOS","Yangambi","Yoko"],
    mosaics=[dict(site=assign_site(p.stem),file=p.name,path=str(p),
                  bytes=p.stat().st_size) for p in rgb_paths],
    shared_annotations={site:str(path) for site,path in ann_site.items()},
    roi=str(ROI_FILE),
    exhaustive_annotations=dict(positive_polygons=str(CIRCLE_LABELS),
        complete_roi_boundaries=str(CIRCLE_ROIS),
        circle_count=int(len(exhaustive_rois00)),
        musanga_polygons=int(len(exhaustive_crowns00)),
        polygon_area_containment_fraction=float(coverage00),
        supervision_rule="outside Musanga polygons = negative ONLY within Circles_ROI; outside ROI = unknown"),
    dsms=[str(p) for p in dsm_paths],
    exclusions=["Luki"],
    remarks=[
       "Six orthomosaics across four site groups; three MOS mosaics use one annotation dataset.",
       "All inputs remain untouched; this notebook inventories files, CRS, label schema and spatial overlap.",
       "ROI modelling extent has been checked only at footprint level; clipping/pixel mask follows in V3/01.",
       "Musanga class coding requires deliberate confirmation prior to model training."
    ])
(OUT/"00_input_manifest.json").write_text(json.dumps(manifest,indent=2))
print("V3 Notebook 00 complete. Manifest:",OUT/"00_input_manifest.json")
print("RGB mosaics:",len(rgb_paths),"| sites:",sorted(set(assign_site(p.stem) for p in rgb_paths)))
print("DSM:",[p.name for p in dsm_paths],"| ROI:",ROI_FILE.name)
print("Next: V3/01 reference polygons and ROI masks; then U-Net development.")
